### RAG Pipelines - Data Ingestion to Vector DB pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\shekh\AppData\Local\Temp\ipykernel_23280\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [3]:
### Read all pdf inside the directory and its subdirectories

def process_pdf_directory(directory_path):

    """Process all PDF files in the specified directory and its subdirectories."""
    all_documents = []
    pdf_dir = Path(directory_path)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))  # Use glob to find all PDF files recursively

    print(f"Found {len(pdf_files)} PDF files in the directory and its subdirectories.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file.name}")
        try:
            pdf_loader = PyPDFLoader(str(pdf_file))
            documents = pdf_loader.load()

            for document in documents:
                # Add metadata
                document.metadata["source"] = str(pdf_file.name)
                document.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}\n\n")
        
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_docs = process_pdf_directory("../data/")


Found 5 PDF files in the directory and its subdirectories.
Processing file: CATApplication.pdf
Loaded 10 pages from CATApplication.pdf


Processing file: FinancialLiterarcyCourse.pdf
Loaded 1 pages from FinancialLiterarcyCourse.pdf


Processing file: ResumeJ.pdf
Loaded 2 pages from ResumeJ.pdf


Processing file: Shubham_Shekhawat_Resume_ANZ.pdf
Loaded 1 pages from Shubham_Shekhawat_Resume_ANZ.pdf


Processing file: SWE_Resume_Template.pdf
Loaded 2 pages from SWE_Resume_Template.pdf


Total documents loaded: 16


In [4]:
all_pdf_docs

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-26T21:13:30+05:30', 'author': 'Sandeep Singh Shekhawat', 'moddate': '2026-08-26T21:13:30+05:30', 'title': 'CAT 2026', 'source': 'CATApplication.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-26T21:13:30+05:30', 'author': 'Sandeep Singh Shekhawat', 'moddate': '2026-08-26T21:13:30+05:30', 'title': 'CAT 2026', 'source': 'CATApplication.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-26T21:13:30+05:30', 'author': 'Sandeep Singh Shekhawat', 'moddate': '2026-08-26T21:13:30+05:30', 'title': 'CAT 2026', 'source': 'CATApplication.pdf', 'total_pages': 10, 'page': 2, 'page_label': '3', 'file_type': 

In [7]:
### Text split get into chunks

def split_documents(documents, chunk_size=100, chunk_overlap=50):
    """Split documents into chunks for better rag performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print("Examole of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunked_documents = split_documents(all_pdf_docs)


Split 16 documents into 120 chunks.
Examole of a chunk:
Content: .......................................................
SANDEEP SINGH SHEKHAWAT
06-09-2026...
Metadata: {'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-3efcf2e67f959db3888d79f73dde2dbd7acb4f8e', 'creationdate': '2026-09-06T13:19:55+05:30', 'moddate': '2026-09-06T13:19:55+05:30', 'source': 'FinancialLiterarcyCourse.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}


In [8]:
chunked_documents

[Document(metadata={'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-3efcf2e67f959db3888d79f73dde2dbd7acb4f8e', 'creationdate': '2026-09-06T13:19:55+05:30', 'moddate': '2026-09-06T13:19:55+05:30', 'source': 'FinancialLiterarcyCourse.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='.......................................................\nSANDEEP SINGH SHEKHAWAT\n06-09-2026'),
 Document(metadata={'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-3efcf2e67f959db3888d79f73dde2dbd7acb4f8e', 'creationdate': '2026-09-06T13:19:55+05:30', 'moddate': '2026-09-06T13:19:55+05:30', 'source': 'FinancialLiterarcyCourse.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'file_type': 'pdf'}, page_content='SANDEEP SINGH SHEKHAWAT\n06-09-2026\n ɱधानमंɬी िवकिसत भारत रोजगार योजना\nकमǺचारी भिवȜय िनिध संगठन'),
 Document(metadata={'producer': 'OpenPDF 1.3.30', 'creator': 'JasperReports Library version 6.20.5-

### Embedding & Vector Store DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\shekh\AI_Project\TradRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer and manages storage in ChromaDB."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initialize the EmbeddingManager with a specified SentenceTransformer model.

            Args:
                model_name (str): The name of the SentenceTransformer model to use for embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentencceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts...
            
            Args:
                texts: List of text strings to embed.
                
            Returns:
                A numnpy array of embeddings with shape (len(texts), embedding_dim).
        """
        if not self.model:
            raise ValueError("Model is not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### Initilixe embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


c:\Users\shekh\AI_Project\TradRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shekh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\shekh\AI_Project\TradRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py

Model loaded successfully, Embedding dimension: 384


### VectorStore

In [14]:
class VectorStore:
    """ Manages a vector store using ChromaDB for storing and retrieving document embeddings. """

    def __init__(self, collection_name: str = "pdf_documents", persist_directory : str = "../data/vetor_store"):
        """
           Initialize the vector store

           Args:
               collection_name: Name of the ChromaDB collection
               persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize  ChromaDB client and collection"""

        try:
            #create persist chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                    name=self.collection_name,
                    metadata={"description":"PDF document embeddings for RAG"}
                    )
            print(f"Vector store initialized collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print("Error initializing vector store: {e}")
            raise

    def add_documents(self, documents:List[Any], embeddings:np.ndarray):
        """ 
            Add documents and their embeddings to the  vector store
            
            Args: 
                documents: List of langchain documents
                embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("No. of documents must no. of embeddings!")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadata = []
        document_txt = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            meta = dict(doc.metadata)
            meta['doc_index'] = i
            meta['context_length'] = len(doc.page_content)
            metadata.append(meta)

            #Document content
            document_txt.append(doc.page_content)

            # Embedding
            embeddings_list.append(embeddings.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadata,
                documents=document_txt,
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e} ")
            raise

vectorstore = VectorStore()
vectorstore

Vector store initialized collection: pdf_documents
Existing documents in collection: 0


In [15]:
### convert text to embeddings
texts = [doc.page_content for doc in chunked_documents]

## Generate embeddings for the chunked documents
embeddings = embedding_manager.generate_embeddings(texts)

## Store into vector db
vectorstore.add_documents(chunked_documents, embeddings)

Generating embeddings for 120 texts...
Generated embeddings with shape: (120, 384)
Adding 120 documents to vector store...
Successfully added 120 documents to vector store.
Total documents in collection: 120
